# Testing integral modules  

We first install our package in editable mode, so that we can import it and test the integral modules.


In [ ]:
%pip install -e /Users/rolandmitric/WORK/GITHUB/master_programming_2026

## Importing the package and setting up the molecule and basis set

```python

In [ ]:
from theochem2026 import Atom, Molecule, BasisSet, Shell, ELEMENT_SYMBOLS
from theochem2026.molecular_integrals import MolecularIntegrals

In [ ]:
benzene_xyz = """6
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""

benzene = Molecule.from_string(benzene_xyz)

In [ ]:
sto3g = BasisSet("sto-3g")
sto3g.get_basis_set(["C", "H", "O", "N", "P"])

ints = MolecularIntegrals(benzene, sto3g)

S_my = ints.overlap_matrix()
T_my = ints.kinetic_matrix()
V_my = ints.nuclear_attraction_matrix()
ERI_my = ints.electron_repulsion_tensor()


### Compare with PySCF (benzene, STO-3G)

PySCF is a widely used quantum chemistry package that will be used to test our code. The reference that needs to be cited when using PySCF is: 

Qiming Sun, Xuebo Li, Rui Pan, Gongjie Bao, Sandeep Paul, Weitang Cai, Yunxuan Feng, Dandan Geng, Zhihao Hao, Shimin Lan, Fei Ma, Yu Mei, Jingsong Qian, Chao Ren, Yihang Sun, Zhenrong Sun, Shuhua Suo, Dingxuan Wang, Bingsen Wei, Shuyu Yang, Sijing Zhang and Qian Peng. PySCF: the Python-based simulations of chemistry framework. Wiley Interdisciplinary Reviews: Computational Molecular Science 2018; 8:e1340. https://doi.org/10.1002/wcms.1340

In [ ]:
import numpy as np
from pyscf import gto

mol_pyscf = gto.M(
    atom=[(atom.symbol, tuple(atom.coord)) for atom in benzene.atoms],
    basis='sto-3g',
    unit='Angstrom',
    verbose=0,
)
S_pyscf = mol_pyscf.intor('int1e_ovlp')
T_pyscf = mol_pyscf.intor('int1e_kin')
V_pyscf = mol_pyscf.intor('int1e_nuc')
ERI_pyscf = mol_pyscf.intor('int2e')

In [ ]:

delta = S_my - S_pyscf
print('S_my shape   :', S_my.shape)
print('S_pyscf shape:', S_pyscf.shape)
print('max |Δ|      :', np.max(np.abs(delta)))
print('||Δ||_F      :', np.linalg.norm(delta))
print('allclose     :', np.allclose(S_my, S_pyscf, atol=1e-10, rtol=1e-5))

In [ ]:
delta = T_my - T_pyscf
print('T_my shape   :', T_my.shape)
print('T_pyscf shape:', T_pyscf.shape)
print('max |Δ|      :', np.max(np.abs(delta)))
print('||Δ||_F      :', np.linalg.norm(delta))
print('allclose     :', np.allclose(T_my, T_pyscf, atol=1e-10, rtol=1e-5))  

In [ ]:
delta = V_my - V_pyscf
print('V_my shape   :', V_my.shape)
print('V_pyscf shape:', V_pyscf.shape)
print('max |Δ|      :', np.max(np.abs(delta)))
print('||Δ||_F      :', np.linalg.norm(delta))
print('allclose     :', np.allclose(V_my, V_pyscf, atol=1e-10, rtol=1e-5)) 

In [ ]:
delta = ERI_my - ERI_pyscf
print('ERI_my shape   :', ERI_my.shape)
print('ERI_pyscf shape:', ERI_pyscf.shape)
print('max |Δ|        :', np.max(np.abs(delta)))
print('||Δ||_F        :', np.linalg.norm(delta))
print('allclose       :', np.allclose(ERI_my, ERI_pyscf, atol=1e-10, rtol=1e-5))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 2, figsize=(10, 10))
axes[0, 0].set_title('Overlap Matrix ')
axes[0, 1].set_title('Overlap Matrix (PySCF)')  
axes[0, 0].imshow(S_my, cmap='viridis')
axes[0, 1].imshow(S_pyscf, cmap='viridis')
axes[1, 0].set_title('Kinetic Matrix')
axes[1, 1].set_title('Kinetic Matrix (PySCF)')
axes[1, 0].imshow(T_my, cmap='viridis')
axes[1, 1].imshow(T_pyscf, cmap='viridis')
axes[2, 0].set_title('Nuclear Attraction Matrix')
axes[2, 1].set_title('Nuclear Attraction Matrix (PySCF)')
axes[2, 0].imshow(V_my, cmap='viridis')
axes[2, 1].imshow(V_pyscf, cmap='viridis')
axes[3, 0].set_title('Electron Repulsion Tensor')
axes[3, 1].set_title('Electron Repulsion Tensor (PySCF)')
axes[3, 0].imshow(ERI_my.reshape(ERI_my.shape[0]*ERI_my.shape[1], ERI_my.shape[2]*ERI_my.shape[3]), cmap='viridis')
axes[3, 1].imshow(ERI_pyscf.reshape(ERI_pyscf.shape[0]*ERI_pyscf.shape[1], ERI_pyscf.shape[2]*ERI_pyscf.shape[3]), cmap='viridis')
plt.tight_layout()
plt.show()